# SurfaceGuard — Notebook 01: Dataset EDA & Mask Visualisation

This notebook performs exploratory analysis of the MVTec AD dataset for industrial surface defect segmentation:

1. Dataset structure overview
2. Class distribution (defect pixel counts)
3. Ground-truth mask visualisation
4. Augmentation demonstration
5. Class imbalance analysis (defect pixels vs background)

> **Dataset note:** This notebook demonstrates with synthetic data if MVTec AD is not downloaded.
> See `data/README.md` for download instructions.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from pathlib import Path

sys.path.insert(0, '..')

np.random.seed(42)
print('Libraries loaded.')

## 1. MVTec AD Category Overview

In [ ]:
# MVTec AD statistics (from the official paper)
mvtec_categories = {
    'bottle':     {'train': 209, 'test_ok': 20, 'test_defect': 83, 'defect_types': ['broken_large', 'broken_small', 'contamination']},
    'cable':      {'train': 224, 'test_ok': 58, 'test_defect': 92, 'defect_types': ['bent_wire', 'cable_swap', 'combined', 'cut_inner_insulation', 'cut_outer_insulation', 'missing_cable', 'poke_insulation']},
    'capsule':    {'train': 219, 'test_ok': 23, 'test_defect': 109,'defect_types': ['crack', 'faulty_imprint', 'poke', 'scratch', 'squeeze']},
    'carpet':     {'train': 280, 'test_ok': 28, 'test_defect': 89, 'defect_types': ['color', 'cut', 'hole', 'metal_contamination', 'thread']},
    'grid':       {'train': 264, 'test_ok': 21, 'test_defect': 57, 'defect_types': ['bent', 'broken', 'glue', 'metal_contamination', 'thread']},
    'hazelnut':   {'train': 391, 'test_ok': 40, 'test_defect': 70, 'defect_types': ['crack', 'cut', 'hole', 'print']},
    'leather':    {'train': 245, 'test_ok': 32, 'test_defect': 92, 'defect_types': ['color', 'cut', 'fold', 'glue', 'poke']},
    'metal_nut':  {'train': 220, 'test_ok': 22, 'test_defect': 93, 'defect_types': ['bent', 'color', 'flip', 'scratch']},
    'pill':       {'train': 267, 'test_ok': 26, 'test_defect': 141,'defect_types': ['color', 'combined', 'contamination', 'crack', 'faulty_imprint', 'pill_type', 'scratch']},
    'screw':      {'train': 320, 'test_ok': 41, 'test_defect': 119,'defect_types': ['manipulated_front', 'scratch_head', 'scratch_neck', 'thread_side', 'thread_top']},
    'tile':       {'train': 230, 'test_ok': 33, 'test_defect': 84, 'defect_types': ['crack', 'glue_strip', 'gray_stroke', 'oil', 'rough']},
    'toothbrush': {'train':  60, 'test_ok':  12,'test_defect': 30, 'defect_types': ['defective']},
    'transistor': {'train': 213, 'test_ok': 60, 'test_defect': 40, 'defect_types': ['bent_lead', 'cut_lead', 'damaged_case', 'misplaced']},
    'wood':       {'train': 247, 'test_ok': 19, 'test_defect': 60, 'defect_types': ['color', 'combined', 'hole', 'liquid', 'scratch']},
    'zipper':     {'train': 240, 'test_ok': 32, 'test_defect': 119,'defect_types': ['broken_teeth', 'combined', 'fabric_border', 'fabric_interior', 'rough', 'split_teeth', 'squeezed_teeth']},
}

# Summary table
print(f'{'Category':<15} {'Train':>7} {'Test OK':>8} {'Test Defect':>12} {'# Types':>8}')
print('-' * 55)
total_train, total_ok, total_def = 0, 0, 0
for cat, info in mvtec_categories.items():
    print(f"{cat:<15} {info['train']:>7} {info['test_ok']:>8} {info['test_defect']:>12} {len(info['defect_types']):>8}")
    total_train += info['train']; total_ok += info['test_ok']; total_def += info['test_defect']
print('-' * 55)
print(f"{'TOTAL':<15} {total_train:>7} {total_ok:>8} {total_def:>12}")

## 2. Training Sample Distribution

In [ ]:
categories = list(mvtec_categories.keys())
train_counts = [mvtec_categories[c]['train'] for c in categories]
test_ok = [mvtec_categories[c]['test_ok'] for c in categories]
test_def = [mvtec_categories[c]['test_defect'] for c in categories]

x = np.arange(len(categories))
fig, ax = plt.subplots(figsize=(16, 5))
bars1 = ax.bar(x - 0.25, train_counts, 0.25, label='Train (normal)', color='steelblue')
bars2 = ax.bar(x,         test_ok,     0.25, label='Test (normal)',  color='mediumseagreen')
bars3 = ax.bar(x + 0.25, test_def,    0.25, label='Test (defect)',  color='tomato')
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=45, ha='right')
ax.set_title('MVTec AD — Sample Count per Category', fontsize=13)
ax.set_ylabel('Number of images')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Total training images (all normal): {total_train}')
print(f'Total test images — normal: {total_ok}, defective: {total_def}')
print(f'Test set defect ratio: {total_def/(total_ok+total_def)*100:.1f}%')

## 3. Synthetic Defect Mask Visualisation

In [ ]:
def make_synthetic_surface(h=256, w=256, defect_type='scratch'):
    """Create a synthetic surface image and corresponding defect mask."""
    # Background texture
    base = np.random.randint(160, 200, (h, w, 3), dtype=np.uint8)
    base += np.random.randint(0, 15, (h, w, 3), dtype=np.uint8)
    mask = np.zeros((h, w), dtype=np.uint8)

    if defect_type == 'scratch':
        x1, y1 = np.random.randint(20, 80), np.random.randint(50, 100)
        x2, y2 = x1 + np.random.randint(80, 160), y1 + np.random.randint(-10, 10)
        cv2.line(base, (x1, y1), (x2, y2), (60, 50, 40), np.random.randint(1, 4))
        cv2.line(mask, (x1, y1), (x2, y2), 255, 3)
    elif defect_type == 'crack':
        pts = np.array([[100, 80], [110, 120], [105, 160], [115, 200]], dtype=np.int32)
        cv2.polylines(base, [pts], False, (50, 40, 30), 2)
        cv2.polylines(mask, [pts], False, 255, 4)
    elif defect_type == 'hole':
        cx, cy = np.random.randint(80, 180), np.random.randint(80, 180)
        r = np.random.randint(10, 25)
        cv2.circle(base, (cx, cy), r, (30, 25, 20), -1)
        cv2.circle(mask, (cx, cy), r, 255, -1)
    elif defect_type == 'contamination':
        for _ in range(np.random.randint(3, 8)):
            cx, cy = np.random.randint(30, 220), np.random.randint(30, 220)
            r = np.random.randint(5, 18)
            cv2.circle(base, (cx, cy), r, (100, 80, 60), -1)
            cv2.circle(mask, (cx, cy), r, 255, -1)

    return base, mask


defect_types = ['scratch', 'crack', 'hole', 'contamination']
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Synthetic Defect Images with Ground-Truth Masks', fontsize=13)

for col, dtype in enumerate(defect_types):
    img, mask = make_synthetic_surface(defect_type=dtype)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # Overlay
    overlay = img_rgb.copy()
    overlay[mask > 0] = (overlay[mask > 0] * 0.5 + np.array([255, 50, 50]) * 0.5).astype(np.uint8)

    axes[0, col].imshow(img_rgb)
    axes[0, col].set_title(f'{dtype}\n(original)', fontsize=10)
    axes[0, col].axis('off')

    axes[1, col].imshow(mask, cmap='hot')
    axes[1, col].set_title('GT mask', fontsize=10)
    axes[1, col].axis('off')

    axes[2, col].imshow(overlay)
    axes[2, col].set_title('overlay', fontsize=10)
    axes[2, col].axis('off')

plt.tight_layout()
plt.show()

## 4. Class Imbalance: Defect Pixels vs Background

In [ ]:
# Simulate defect pixel fractions for different defect types (approximate real MVTec values)
defect_pixel_fractions = {
    'scratch': np.random.uniform(0.005, 0.025, 20),
    'crack':   np.random.uniform(0.010, 0.040, 20),
    'hole':    np.random.uniform(0.020, 0.060, 20),
    'contamination': np.random.uniform(0.008, 0.035, 20),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
ax = axes[0]
labels = list(defect_pixel_fractions.keys())
data = [defect_pixel_fractions[k] for k in labels]
bp = ax.boxplot(data, labels=labels, patch_artist=True)
colors = ['#e74c3c', '#e67e22', '#3498db', '#2ecc71']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_title('Defect Pixel Fraction by Defect Type', fontsize=11)
ax.set_ylabel('Fraction of image pixels (defect)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.1f}%'))
ax.grid(axis='y', alpha=0.3)

# Class imbalance pie for a typical defect image
ax = axes[1]
typical_defect_pct = 0.02  # ~2% defect pixels
ax.pie([1 - typical_defect_pct, typical_defect_pct],
       labels=['Background pixels\n(98%)', 'Defect pixels\n(2%)'],
       colors=['#bdc3c7', '#e74c3c'],
       startangle=90,
       autopct='%1.1f%%',
       explode=[0, 0.1])
ax.set_title('Typical Class Imbalance in a Defective Image', fontsize=11)

plt.suptitle('Why BCEDiceLoss? Extreme Class Imbalance Motivates Dice-Aware Training', fontsize=12)
plt.tight_layout()
plt.show()

print('Defect pixels are typically 1–6% of total image pixels.')
print('Pure BCE loss would learn to predict all-background (trivial solution).')
print('Dice loss directly optimises mask overlap → handles class imbalance.')

## 5. Paired Augmentation Demonstration

In [ ]:
try:
    from surfaceguard.data.transforms import PairedSegmentationTransforms
    import torch
    from PIL import Image

    transforms = PairedSegmentationTransforms(image_size=256, is_train=True)
    img, mask = make_synthetic_surface(defect_type='scratch')
    img_pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    mask_pil = Image.fromarray(mask)

    fig, axes = plt.subplots(2, 5, figsize=(18, 7))
    fig.suptitle('Paired Augmentation — Same Transform Applied to Image AND Mask', fontsize=12)

    axes[0, 0].imshow(np.array(img_pil))
    axes[0, 0].set_title('Original Image', fontsize=9)
    axes[0, 0].axis('off')
    axes[1, 0].imshow(np.array(mask_pil), cmap='hot')
    axes[1, 0].set_title('Original Mask', fontsize=9)
    axes[1, 0].axis('off')

    for i in range(1, 5):
        aug_img, aug_mask = transforms(img_pil, mask_pil)
        if hasattr(aug_img, 'numpy'):
            aug_img_np = aug_img.permute(1,2,0).numpy()
            aug_img_np = ((aug_img_np - aug_img_np.min()) / (aug_img_np.max() - aug_img_np.min() + 1e-8) * 255).astype(np.uint8)
        else:
            aug_img_np = np.array(aug_img)
        if hasattr(aug_mask, 'numpy'):
            aug_mask_np = aug_mask.squeeze().numpy()
        else:
            aug_mask_np = np.array(aug_mask)

        axes[0, i].imshow(aug_img_np)
        axes[0, i].set_title(f'Augmented #{i}', fontsize=9)
        axes[0, i].axis('off')
        axes[1, i].imshow(aug_mask_np, cmap='hot')
        axes[1, i].set_title(f'Mask #{i}', fontsize=9)
        axes[1, i].axis('off')

    plt.tight_layout()
    plt.show()
    print('Paired augmentation: spatial transforms applied identically to image and mask.')

except ImportError as e:
    print(f'SurfaceGuard transforms not available: {e}')
    print('Run: pip install -e .. from the project root to enable this section.')

## Summary

| Finding | Implication |
|---------|------------|
| MVTec AD: 15 categories, 73 defect types | Per-category models vs. unified model trade-off |
| Training images are ALL defect-free | PatchCore uses this; supervised models need separate defect data |
| Defect pixels: ~1–6% of image | BCEDiceLoss necessary; pure BCE learns trivial solution |
| Diverse defect shapes (scratch, hole, blob) | Pixel-level segmentation required; bounding boxes insufficient |

**Next:** Notebook 02 compares U-Net vs DeepLabV3+ architectures on synthetic data.